# 🏀 NBA Three-Pointer Evolution by Decade
### How the game changed from 1946 to 2026

In this notebook we load a real NBA dataset, group it by decade, and explore how the **3-pointer** — and the game around it — evolved over 80 years.

**Libraries used:** `pandas`, `numpy`, `matplotlib`, `pyspark`  
**Dataset:** `TeamStatistics.csv` — one row per team per game (~145,000 rows)

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("agg")
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Libraries loaded")

In [ ]:
# ── Cell 2: Load data from Google Cloud Storage ─────────────────────────────
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("NBA_Viz").getOrCreate()
sdf = spark.read.option("header","true").option("inferSchema","true").csv("gs://hw8-cs131-omajano-1845-bkt/data/TeamStatistics.csv")
df = sdf.toPandas()
spark.stop()

print(f"Rows   : {len(df):,}")
print(f"Columns: {len(df.columns)}")
print()
print("Data loaded from GCS")
df.head(3)


In [ ]:
# ── Cell 3: Clean & add helper columns ───────────────────────────────────────
# Parse the date column so we can extract the year
df["gameDate"] = pd.to_datetime(df["gameDateTimeEst"])
df["season"]   = df["gameDate"].dt.year

# Assign a decade label: 1946 → "1940s", 1985 → "1980s", etc.
df["decade"]      = (df["season"] // 10) * 10
df["decadeLabel"] = df["decade"].astype(str) + "s"

# Share of shot attempts that are 3-pointers (avoids divide-by-zero)
df["3PA_rate"] = np.where(
    df["fieldGoalsAttempted"] > 0,
    df["threePointersAttempted"] / df["fieldGoalsAttempted"],
    np.nan
)

# Points contributed by 3-pointers vs. 2-pointers vs. free throws
df["pts_from_3"]  = df["threePointersMade"] * 3
df["pts_from_2"]  = (df["fieldGoalsMade"] - df["threePointersMade"]) * 2
df["pts_from_FT"] = df["freeThrowsMade"]

print("Columns added: season, decade, decadeLabel, 3PA_rate, pts_from_3/2/FT")
print(f"Season range  : {df['season'].min()} – {df['season'].max()}")

In [ ]:
# ── Cell 4: Build filtered datasets & decade-level summary ──────────────────
# Filter to games where a 3-pointer was actually attempted (removes pre-3PT era NaNs)
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()

decade_stats = df_3pt.groupby("decadeLabel").agg(
    Games         = ("gameId",                  "count"),
    Avg_3PA       = ("threePointersAttempted",  "mean"),
    Avg_3PM       = ("threePointersMade",        "mean"),
    Avg_3P_pct    = ("threePointersPercentage",  "mean"),
    Avg_FGA       = ("fieldGoalsAttempted",      "mean"),
    Avg_FTA       = ("freeThrowsAttempted",      "mean"),
    Avg_FTM       = ("freeThrowsMade",           "mean"),
    Avg_PTS       = ("teamScore",                "mean"),
    Avg_AST       = ("assists",                  "mean"),
    Avg_TOV       = ("turnovers",                "mean"),
    Avg_steals    = ("steals",                   "mean"),
    Avg_blocks    = ("blocks",                   "mean"),
    Avg_rebounds  = ("reboundsTotal",            "mean"),
).round(2)

decade_stats["Avg_3P_pct"]  = (decade_stats["Avg_3P_pct"]  * 100).round(1)
decade_stats["Avg_3PA_rate"]= (decade_stats["Avg_3PA"] / decade_stats["Avg_FGA"] * 100).round(1)
decade_stats["pts_from_3"]  = (decade_stats["Avg_3PM"] * 3).round(2)

print(f"3PT data: {df_3pt['season'].min()} – {df_3pt['season'].max()}")
print(f"Decades : {decade_stats.index.tolist()}")
print(f"No NaNs in key cols: {decade_stats[['Avg_3PA','Avg_3PM','Avg_3P_pct','Avg_FTA','Avg_steals']].isna().sum().sum() == 0}")
decade_stats


---
### Table 1 — 3-Pointer Stats Only

In [ ]:
# ── Table 1: 3PT focused ──────────────────────────────────────────────────────
table1 = decade_stats[["Games", "Avg_3PA", "Avg_3PM", "Avg_3P_pct", "Avg_3PA_rate"]].copy()
table1.columns = ["Games", "Avg 3PA", "Avg 3PM", "3P%", "3PA/FGA %"]
print("Table 1 — Three-Pointer Volume & Efficiency by Decade")
print("=" * 55)
table1

---
### Table 2 — Scoring & Supporting Factors

In [ ]:
# ── Table 2: Supporting factors ─────────────────────────────────────────────
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()

table2 = df_3pt.groupby("decadeLabel").agg(
    Avg_PTS = ("teamScore",           "mean"),
    Avg_FGA = ("fieldGoalsAttempted", "mean"),
    Avg_FTA = ("freeThrowsAttempted", "mean"),
    Avg_AST = ("assists",             "mean"),
    Avg_TOV = ("turnovers",           "mean"),
    Avg_STL = ("steals",              "mean"),
    Avg_BLK = ("blocks",              "mean"),
).round(2)

table2.columns = ["Avg PTS", "Avg FGA", "Avg FTA", "Avg AST", "Avg TOV", "Avg STL", "Avg BLK"]
print("Table 2 — Scoring & Supporting Factors by Decade")
print("=" * 55)
table2


---
### Table 3 — Season-Level Summary (every year)

In [ ]:
# ── Table 3: Season-level ─────────────────────────────────────────────────────
season_stats = df.groupby("season").agg(
    Games      = ("gameId",                 "count"),
    Avg_3PA    = ("threePointersAttempted", "mean"),
    Avg_3PM    = ("threePointersMade",       "mean"),
    Avg_3P_pct = ("threePointersPercentage", "mean"),
    Avg_PTS    = ("teamScore",               "mean"),
    Avg_AST    = ("assists",                 "mean"),
    Avg_PITP   = ("pointsInThePaint",        "mean"),
).round(2)
season_stats["Avg_3P_pct"] = (season_stats["Avg_3P_pct"] * 100).round(1)

print("Table 3 — Per-Season Averages (3PT era, 1980+)")
print("=" * 55)
season_stats[season_stats.index >= 1980]

---
### Table 4 — Pre-Analytics vs Analytics Era Comparison

In [ ]:
# ── Table 4: Era comparison ───────────────────────────────────────────────────
modern = df[df["season"] >= 1980].copy()
modern["era"] = np.where(modern["season"] < 2015,
                          "Pre-Analytics (1980–2014)",
                          "Analytics Era (2015–2026)")

table4 = modern.groupby("era").agg(
    Games      = ("gameId",                 "count"),
    Avg_3PA    = ("threePointersAttempted", "mean"),
    Avg_3PM    = ("threePointersMade",       "mean"),
    Avg_3P_pct = ("threePointersPercentage", "mean"),
    Avg_PTS    = ("teamScore",               "mean"),
    Avg_PITP   = ("pointsInThePaint",        "mean"),
    Avg_FTA    = ("freeThrowsAttempted",     "mean"),
).round(2)
table4["Avg_3P_pct"] = (table4["Avg_3P_pct"] * 100).round(1)
table4.columns = ["Games", "Avg 3PA", "Avg 3PM", "3P%", "Avg PTS", "Avg PITP", "Avg FTA"]

print("Table 4 — Pre-Analytics vs Analytics Era")
print("=" * 55)
table4

---
### Table 5 — Decade-over-Decade % Growth in 3PA

In [ ]:
# ── Table 5: Growth rates ─────────────────────────────────────────────────────
growth = decade_stats.loc[decade_stats["Avg_3PA"] > 0, ["Avg_3PA", "Avg_3PM", "Avg_PTS"]].copy()
growth["3PA Growth %"] = growth["Avg_3PA"].pct_change().mul(100).round(1)
growth["3PM Growth %"] = growth["Avg_3PM"].pct_change().mul(100).round(1)
growth["PTS Growth %"] = growth["Avg_PTS"].pct_change().mul(100).round(1)
growth.columns = ["Avg 3PA", "Avg 3PM", "Avg PTS",
                   "3PA Growth %", "3PM Growth %", "PTS Growth %"]

print("Table 5 — Decade-over-Decade Growth (3PT era only)")
print("=" * 55)
growth

---
### Table 6 — Correlation Matrix (3PT era, season-level)

In [ ]:
# ── Table 6: Correlation matrix ───────────────────────────────────────────────
corr_cols = ["Avg_3PA", "Avg_3PM", "Avg_PTS", "Avg_AST", "Avg_PITP"]
corr_labels = ["3PA", "3PM", "PTS", "AST", "PITP"]

corr_data = season_stats.loc[season_stats.index >= 1980, corr_cols].copy()
corr_data.columns = corr_labels
corr_matrix = corr_data.corr().round(2)

print("Table 6 — Pearson Correlation Matrix (Season-Level, 1980–2026)")
print("Values close to 1 or -1 = strong relationship")
print("=" * 55)
corr_matrix

---
## 📊 Visualizations

### Figure 1 — Average 3-Pointers Attempted Per Game by Decade

In [ ]:
# ── Figure 1: Avg 3PA per Game by Decade ────────────────────────────────────
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()
dec = df_3pt.groupby("decadeLabel")["threePointersAttempted"].mean().round(2)

labels  = dec.index.tolist()
vals    = dec.values
decades = [int(l.replace("s","")) for l in labels]
colors  = ["#E63946" if d >= 2010 else "#457B9D" for d in decades]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, vals, color=colors, edgecolor="white", linewidth=0.7)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#457B9D", label="Early 3PT era (1986–2009)"),
    Patch(color="#E63946", label="Modern era (2010s+)"),
], loc="upper left")

ax.set_title("Average 3-Pointers Attempted Per Team-Game by Decade", fontsize=13)
ax.set_xlabel("Decade")
ax.set_ylabel("Avg 3PA per Game")
ax.set_ylim(0, vals.max() * 1.2)
plt.tight_layout()
plt.show()


---
### Figure 2 — Season-by-Season 3PA & 3PM Trend (1980–2026)

In [ ]:
# ── Figure 2: Season-by-season 3PA & 3PM trend ──────────────────────────────
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()
season_3pt = df_3pt.groupby("season").agg(
    Avg_3PA = ("threePointersAttempted", "mean"),
    Avg_3PM = ("threePointersMade",       "mean"),
).round(2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(season_3pt.index, season_3pt["Avg_3PA"], alpha=0.15, color="#E63946")
ax.plot(season_3pt.index, season_3pt["Avg_3PA"],
        color="#E63946", linewidth=2.2, marker="o", markersize=3, label="Avg 3PA")
ax.plot(season_3pt.index, season_3pt["Avg_3PM"],
        color="#2a9d8f", linewidth=1.8, linestyle="--", marker="s", markersize=3, label="Avg 3PM")

for yr, note in [(1995, "Line closer"), (1997, "Line back"), (2016, "Analytics boom")]:
    if yr in season_3pt.index:
        y = season_3pt.loc[yr, "Avg_3PA"]
        ax.annotate(note, xy=(yr, y), xytext=(yr+1, y+3),
                    fontsize=8, color="#444",
                    arrowprops=dict(arrowstyle="->", color="#aaa", lw=0.8))

ax.set_title("Season-by-Season Avg 3PA and 3PM Per Team-Game (1986–2026)", fontsize=13)
ax.set_xlabel("Season")
ax.set_ylabel("Avg per Game")
ax.legend()
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.show()


---
### Figure 3 — Scoring Composition by Decade (2PT vs 3PT vs Free Throws)

In [ ]:
# ── Figure 3: Scoring composition by decade ─────────────────────────────────
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()
comp = df_3pt.groupby("decadeLabel").agg(
    fg_made  = ("fieldGoalsMade",    "mean"),
    tp_made  = ("threePointersMade", "mean"),
    ft_made  = ("freeThrowsMade",    "mean"),
    total    = ("teamScore",         "mean"),
).round(2)
comp["pts_2"] = (comp["fg_made"] - comp["tp_made"]) * 2
comp["pts_3"] = comp["tp_made"] * 3

x = np.arange(len(comp))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, comp["pts_2"],  label="2PT Field Goals", color="#457B9D")
ax.bar(x, comp["pts_3"],  label="3PT Field Goals", color="#E63946", bottom=comp["pts_2"])
ax.bar(x, comp["ft_made"],label="Free Throws",     color="#a8dadc",
       bottom=comp["pts_2"] + comp["pts_3"])
ax.plot(x, comp["total"], color="black", marker="o",
        linewidth=2, markersize=5, label="Actual Avg PTS", zorder=5)
ax.set_xticks(x)
ax.set_xticklabels(comp.index)
ax.set_title("Scoring Composition Per Team-Game by Decade", fontsize=13)
ax.set_ylabel("Points")
ax.set_xlabel("Decade")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


---
### Figure 4 — Paint Points vs 3PT Points Over Time

In [ ]:
# ── Figure 4: Points from 3s vs Free Throw Attempts by Decade ───────────────
# As 3PA rose, teams got to the free throw line less — a clear trade-off
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()
dec_ft = df_3pt.groupby("decadeLabel").agg(
    pts_from_3 = ("threePointersMade",   "mean"),
    avg_fta    = ("freeThrowsAttempted", "mean"),
).round(2)
dec_ft["pts_from_3"] = (dec_ft["pts_from_3"] * 3).round(2)

x = np.arange(len(dec_ft))
w = 0.38
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, dec_ft["pts_from_3"], w,
               label="Points from 3-Pointers", color="#E63946")
bars2 = ax.bar(x + w/2, dec_ft["avg_fta"],    w,
               label="Free Throws Attempted",  color="#457B9D")

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.2,
                f"{h:.1f}", ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(dec_ft.index)
ax.set_title("Points from 3-Pointers vs Free Throw Attempts by Decade\n"
             "(As 3PA rose, teams got to the line less)", fontsize=12)
ax.set_ylabel("Avg per Game")
ax.set_xlabel("Decade")
ax.legend()
plt.tight_layout()
plt.show()


---
### Figure 5 — Supporting Factors: Assists, Turnovers & Fast Breaks

In [ ]:
# ── Figure 5: Steals, Blocks & Rebounds by Decade (all 1980+) ───────────────
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = [
    ("steals",        "Avg Steals per Game",         "#2a9d8f"),
    ("blocks",        "Avg Blocks per Game",          "#e9c46a"),
    ("reboundsTotal", "Avg Total Rebounds per Game",  "#f4a261"),
]
for ax, (col, title, color) in zip(axes, metrics):
    by_dec = df_3pt.groupby("decadeLabel")[col].mean().round(2)
    ax.bar(by_dec.index, by_dec.values, color=color, edgecolor="white", linewidth=0.6)
    for i, v in enumerate(by_dec.values):
        ax.text(i, v + 0.05, f"{v:.1f}", ha="center", fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Decade")
    ax.tick_params(axis="x", rotation=35)
    ax.set_ylim(0, by_dec.values.max() * 1.2)

plt.suptitle("Defensive & Rebounding Stats by Decade (1986+)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
### Figure 6 — 3PA vs Team Scoring: Scatter + Regression Line

In [ ]:
# ── Figure 6: Did Shooting More 3s Lead to More Points? ─────────────────────
df_3pt = df[(df["threePointersAttempted"].notna()) & (df["threePointersAttempted"] > 0)].copy()
by_dec = df_3pt.groupby("decadeLabel").agg(
    Avg_PTS = ("teamScore",              "mean"),
    Avg_3PA = ("threePointersAttempted", "mean"),
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: team score by decade
ax = axes[0]
colors = ["#f4d35e","#f4d35e","#f79256","#f79256","#E63946"]
bars = ax.bar(by_dec.index, by_dec["Avg_PTS"],
              color=colors, edgecolor="white", linewidth=0.7)
for bar, val in zip(bars, by_dec["Avg_PTS"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}", ha="center", fontsize=9)
ax.set_title("Avg Team Score by Decade", fontsize=12)
ax.set_xlabel("Decade")
ax.set_ylabel("Avg Points per Game")
ax.set_ylim(90, by_dec["Avg_PTS"].max() * 1.1)

# Right: 3PA bars + scoring line overlay
ax2 = axes[1]
x = np.arange(len(by_dec))
w = 0.5
ax2.bar(x, by_dec["Avg_3PA"], w, label="Avg 3PA", color="#E63946", alpha=0.85)
ax2_twin = ax2.twinx()
ax2_twin.plot(x, by_dec["Avg_PTS"], color="black", marker="o",
              linewidth=2.2, markersize=7, label="Avg PTS")
ax2_twin.set_ylabel("Avg Points per Game")
ax2_twin.set_ylim(90, by_dec["Avg_PTS"].max() * 1.1)
ax2.set_xticks(x)
ax2.set_xticklabels(by_dec.index)
ax2.set_title("3PA Volume vs Team Scoring by Decade", fontsize=12)
ax2.set_xlabel("Decade")
ax2.set_ylabel("Avg 3PA per Game")
lines1, lab1 = ax2.get_legend_handles_labels()
lines2, lab2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, lab1 + lab2, loc="upper left")

plt.suptitle("Did Shooting More 3s Lead to More Points?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
